Snow Depth Analysis Notebook
Made by Zane Pederson

In [95]:
from pathlib import Path #to access data by path
import topotoolbox as ttb #to handle data
import matplotlib.pyplot as plt #for plotting
import numpy as np #for handling large amounts of data (arrays not lists)
from datetime import datetime as dt #to process the date of dems
import rioxarray as rxr

A function that filters out improper Lidar depths from matrix and returns a np.array. (Array will contain np.nan, which will need to be filtered for plotting).

In [96]:
#filter out any points less than -0.1 or greater than 5, if a remaining value is less than 0 it will be set to zero
def filterLidarDepth(depths):
    depths[depths < -0.1] = np.nan 
    depths[depths > 5.0] = np.nan
    depths[depths < 0] = 0 
    return depths

Construct list of snow depth dems after filtering out improper Lidar depths.

In [97]:
rasterFolder = Path('Snow Depth Rasters')
dems = [] #snow depth dems

#loop through each file in snow rasters folder and construct array of dems
for file in Path("Snow Depth Rasters").iterdir():
    #ensure file is .tif
    if file.is_file() and file.suffix.lower() == '.tif':
        dem = ttb.read_tif(file)
        filterLidarDepth(dem.z)
        dems.append(dem)

A function that resamples thisDem to match the resolution of targetDem and returns a matrix of thisDem's values.

In [98]:
def resample(thisDem, targetDem): #resample thisDem to match targetDem
    #open both as xarray objectsS
    thisRx = rxr.open_rasterio(thisDem.path)
    targetRx = rxr.open_rasterio(targetDem.path)
    #resample thisRx to match resolution of targetRx
    demResampled = thisRx.rio.reproject_match(targetRx)
    #flatten and return thisDem
    return demResampled.values.ravel().astype(float)

Here we create a list of independent variables that we will plot in relation to snow depth. This includes: Burn Severity, Slope, Aspect, and Northness.

In [ ]:
bsDem = ttb.read_tif(r"NonDepthData\MTBS_Resampled_1m.tif") #raster containing burn severity data
sfDem = ttb.read_tif(r"NonDepthData\DEM_063025_SnowFree.tif") #raster of snow free slope (used for slope, aspect, & northness)

#get slope dem
slopeData = sfDem.gradient8(unit='degree').z #create a matrix
slopeData[slopeData > 80] = np.nan #filter out angles over 80 degrees
slopeData[slopeData < 0] = np.nan #filter out angles less than 0 degrees
slopeDem = sfDem.duplicate_with_new_data(slopeData) #create a new dem containing slope as z

#get aspect dem
aspectData = sfDem.aspect().z
aspectData[aspectData > 360] = np.nan #filter out angles over 360 degrees
aspectData[aspectData < 0] = np.nan #filter out angles less than 0 degrees
aspectDem = sfDem.duplicate_with_new_data(aspectData) 

#get northness dem
northnessData = np.cos(aspectData) * np.sin(slopeData)
northnessDem = sfDem.duplicate_with_new_data(northnessData)

#set names
bsDem.name = "Burn Severity"
sfDem.name = "Snow Free Slope"
slopeDem.name = "Slope"
aspectDem.name = "Aspect"
northnessDem.name = "Northness"

iVarDems = [] #dems of indepent variables ex: burn severity

#add dems to our list
#iVarDems.append(bsDem)
#iVarDems.append(sfDem)
#iVarDems.append(slopeDem)
#iVarDems.append(aspectDem)
#iVarDems.append(northnessDem)

Function that takes the raw name of a dem and processes it into a readable format.

In [101]:
def processDemName(rawName : str):
    rawDate = rawName.split("_")[1] #get date string from name (will take whatever string comes after the first underscore)
    date = dt.strptime(rawDate, "%m%d%y").date() #get date object from string
    name = "Lidar drone data - " + date.strftime("%m/%d/%y") #convert date object back to string
    return name

Create a box plot based on parameters: data (array of numerical values), title (plot title), dataLabel (label describing the numerical data), and criteriaLabel (special criteria this dataset meets).

In [102]:
def makeBoxPlot(data : np.array, title : str, dataLabel : str, criteriaLabel : str):
    if len(data) <= 1: return

    #Setup plot
    plt.figure(figsize=(8, 4))
    plt.boxplot(data)

    #Setup labels
    plt.title(title)
    plt.xlabel(criteriaLabel)
    plt.ylabel(dataLabel)

    #Show plot
    plt.show()

Create a scatter plot based on parameters: xData, yData, title, xLabel, and yLabel

In [103]:
def makeScatterPlot(xData : np.array, yData : np.array, title : str, xLabel : str, yLabel : str):
    
    #Setup graph
    plt.figure(figsize=(8, 4))
    plt.scatter(xData, yData)

    #Setup labels
    plt.title(title)
    plt.xlabel(xLabel)
    plt.ylabel(yLabel)

    #Show graph
    plt.show()

Process snow depth data and make plots.

In [ ]:
for dem in dems:

    # -------------Box plot ---------------
    # make box plot of depths
    if(False):
        depths = dem.z.ravel()
        depths = filterLidarDepth(depths)
        depthsClean = depths[~np.isnan(depths)]
        makeBoxPlot(depthsClean, processDemName(dem.name), "Snow Depth", "All data (Outliers removed)")

    # -------------Scatter plot -----------
    # #make scatter plot of depths against all independent variables
    if(True):
        for iVarDem in iVarDems:

            #resample
            depths = resample(dem, iVarDem)

            #filter
            depths = filterLidarDepth(depths)
            
            #get iVarData
            iVarData = iVarDem.z.ravel()

            if len(depths) == len(iVarData):
                #filter out any points where depths or iVarData is nan
                validMask = ~np.isnan(depths) & ~np.isnan(iVarData)
                makeScatterPlot(depths[validMask], iVarData[validMask], processDemName(dem.name), "Snow Depth", iVarDem.name)
            else:
                print("RESOLUTIONS DID NOT MATCH: " + dem.name)
                print("depths-size:" + str(dem.shape))
                print("independent-size: " + str(iVarDem.shape))